# AF2·04 — The Evoformer Block + Distogram Head

**Mechanism of the day:** put the trunk together. You have the MSA-side attention
(rung 02) and the pair-side triangle operations (rung 03); now wire them into a single
**Evoformer block**, stack it, and read the first genuinely *geometric* prediction out
of the pair representation — a **distance histogram**.

An Evoformer block does four things, in order, and the order is the point:

1. **MSA reasons about itself** — row attention (biased by the pair rep) and column
   attention update the MSA representation `m`.
2. **MSA writes to pair** — the outer-product mean turns the freshly-updated MSA into a
   contribution to the pair representation `z`. This is the communication step.
3. **Pair reasons about itself** — triangle multiplicative update (and, in the full
   model, triangle attention) enforce geometric consistency on `z`.
4. **Transitions** — small per-element MLPs on each.

Repeat, and `m` and `z` refine each other: the MSA tells the pair which residues
coevolve; the pair tells the MSA which residues to compare; the triangle ops keep the
pair geometrically sane. After the trunk, a **distogram head** decodes `z[i,j]` into a
probability distribution over binned `Cα–Cα` **distances** — not a yes/no contact, a
full distance distribution. That distribution is exactly what the structure module
(rung 05+) will turn into 3D coordinates.

Our payoff makes a specific point: predicting a *coherent distance map* is where the
triangle machinery earns its keep. A model that scores each pair in isolation cannot —
distances must obey the triangle inequality, which is a statement about triples. You
will watch the full trunk beat a per-pair baseline decisively on distance prediction.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions
at the bottom. Trains in ~80s on a laptop CPU (two models).

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'
Q = 20; L = 24; Nseq = 200

# ---- toy 3D proteins: a compact, folding chain (given) ----
def make_structure():
    pos = [np.zeros(3)]; d = rng.normal(0, 1, 3); d /= np.linalg.norm(d)
    for _ in range(L - 1):
        d = d + rng.normal(0, 0.5, 3); d /= np.linalg.norm(d)     # persistence -> smooth chain
        pos.append(pos[-1] + d - 0.03 * pos[-1])                  # gentle central pull -> folds
    X = np.array(pos); X -= X.mean(0)
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    return X, D

DIST_BINS = np.linspace(1.5, 9.0, 15)          # 16 distance bins (last is a catch-all)
BIN_CENTERS = torch.tensor(np.concatenate(
    [[DIST_BINS[0] - 0.5], (DIST_BINS[:-1] + DIST_BINS[1:]) / 2, [DIST_BINS[-1] + 0.5]]),
    dtype=torch.float32)
IS_CONTACT_BIN = BIN_CENTERS < 2.5           # which of the 16 bins count as a contact

def make_example():
    '''One toy protein: (MSA, APC-coevolution seed, distogram-bin target, contact map).'''
    X, D = make_structure()
    Cmat = (D < 2.5) & (np.abs(np.arange(L)[:, None] - np.arange(L)[None]) >= 3)
    con = set(map(tuple, np.argwhere(np.triu(Cmat))))
    prof = np.array([rng.dirichlet(np.ones(Q) * 0.8) for _ in range(L)])
    perm = {p: rng.permutation(Q) for p in con}
    m = np.zeros((Nseq, L), int)
    for p in range(L): m[:, p] = rng.choice(Q, size=Nseq, p=prof[p])
    block = rng.choice(L, size=8, replace=False)          # phylogenetic clade confound
    cons = [{c: int(rng.integers(Q)) for c in block} for _ in range(4)]
    cl = rng.integers(4, size=Nseq)
    for c in block:
        for g in range(4):
            h = (cl == g) & (rng.random(Nseq) < 0.9); m[h, c] = cons[g][c]
    for (i, j) in sorted(con):
        cp = rng.random(Nseq) < 0.8; m[cp, j] = perm[(i, j)][m[cp, i]]
    # APC coevolution seed (rung 01)
    oh = np.eye(Q)[m]; ps = 0.5; fi = (oh.sum(0) + ps) / (Nseq + Q * ps); MI = np.zeros((L, L))
    for i in range(L):
        for j in range(i + 1, L):
            fij = (oh[:, i].T @ oh[:, j] + ps / Q) / (Nseq + ps); fij /= fij.sum()
            MI[i, j] = MI[j, i] = (fij * np.log(fij / (np.outer(fi[i], fi[j]) + 1e-12) + 1e-12)).sum()
    mm = MI.copy(); np.fill_diagonal(mm, 0); col = mm.sum(1, keepdims=True) / (L - 1)
    a = mm[~np.eye(L, dtype=bool)].mean(); APC = mm - (col @ col.T) / a; np.fill_diagonal(APC, 0)
    return m, APC.astype(np.float32), np.digitize(D, DIST_BINS).astype(np.int64), Cmat

def build(n):
    M, Z, Dg, Cc = [], [], [], []
    for _ in range(n):
        m, z, dg, c = make_example(); M.append(m); Z.append(z); Dg.append(dg); Cc.append(c)
    return (torch.tensor(np.array(M)), torch.tensor(np.array(Z)),
            torch.tensor(np.array(Dg)), torch.tensor(np.array(Cc)))

t0 = time.time()
Mtr, Ztr, Dtr, Ctr = build(300); Mte, Zte, Dte, Cte = build(40)
UM = torch.triu(torch.ones(L, L), diagonal=3).bool()
print('%d train + %d test proteins in %.1fs | %.1f contacts/protein'
      % (len(Mtr), len(Mte), time.time() - t0, Cte.sum().item() / 2 / len(Cte)))

## Part 1 — the pieces (given), and symmetry (yours)

The module classes below are exactly the operations you built in rungs 02–03, wrapped
as layers: `RowAttnPairBias`, `ColAttn`, `OuterProductMean`, `TriMul` (triangle
multiplicative update), and a `PairTransition` MLP. Read them — there is nothing new,
only plumbing.

One genuinely new requirement: the pair representation must stay **symmetric**, because
`z[i,j]` and `z[j,i]` describe the same physical relationship and the distance between
`i` and `j` is symmetric. We enforce it after the pair updates.

In [ ]:
def row_attention(q, k, v, pb): d = q.size(-1); return F.softmax(q @ k.transpose(-2,-1)/math.sqrt(d)+pb[None], -1) @ v
def column_attention(q, k, v):  d = q.size(-1); return F.softmax(q @ k.transpose(-2,-1)/math.sqrt(d), -1) @ v
def triangle_update(a, b):      return torch.einsum('ikc,jkc->ijc', a, b) + torch.einsum('kic,kjc->ijc', a, b)

class RowAttnPairBias(nn.Module):
    def __init__(s, cm, cz, h=4):
        super().__init__(); s.h, s.d = h, cm // h
        s.q=nn.Linear(cm,cm,bias=False); s.k=nn.Linear(cm,cm,bias=False); s.v=nn.Linear(cm,cm,bias=False)
        s.b=nn.Linear(cz,h,bias=False); s.g=nn.Linear(cm,cm); s.o=nn.Linear(cm,cm); s.ln=nn.LayerNorm(cm)
    def forward(s, m, z):
        N,Ll,cm=m.shape; x=s.ln(m); sp=lambda t:t.view(N,Ll,s.h,s.d).permute(0,2,1,3)
        out=row_attention(sp(s.q(x)),sp(s.k(x)),sp(s.v(x)),s.b(z).permute(2,0,1)).permute(0,2,1,3).reshape(N,Ll,cm)
        return m + s.o(out*torch.sigmoid(s.g(x)))
class ColAttn(nn.Module):
    def __init__(s, cm, h=4):
        super().__init__(); s.h,s.d=h,cm//h
        s.q=nn.Linear(cm,cm,bias=False); s.k=nn.Linear(cm,cm,bias=False); s.v=nn.Linear(cm,cm,bias=False)
        s.g=nn.Linear(cm,cm); s.o=nn.Linear(cm,cm); s.ln=nn.LayerNorm(cm)
    def forward(s, m):
        mt=s.ln(m).transpose(0,1); Ll,N,cm=mt.shape; sp=lambda t:t.view(Ll,N,s.h,s.d).permute(0,2,1,3)
        out=column_attention(sp(s.q(mt)),sp(s.k(mt)),sp(s.v(mt))).permute(0,2,1,3).reshape(Ll,N,cm)
        return m + (s.o(out*torch.sigmoid(s.g(mt)))).transpose(0,1)
class OuterProductMean(nn.Module):
    def __init__(s, cm, cz): super().__init__(); s.ln=nn.LayerNorm(cm); s.proj=nn.Linear(cm*cm, cz)
    def forward(s, m):
        x=s.ln(m); opm=torch.einsum('nic,njd->ijcd', x, x)/x.shape[0]
        return s.proj(opm.reshape(L, L, -1))
class TriMul(nn.Module):
    def __init__(s, c):
        super().__init__(); s.ln=nn.LayerNorm(c); s.lno=nn.LayerNorm(c)
        s.a=nn.Linear(c,c); s.b=nn.Linear(c,c); s.ag=nn.Linear(c,c); s.bg=nn.Linear(c,c); s.o=nn.Linear(c,c); s.og=nn.Linear(c,c)
    def forward(s, z):
        zz=s.ln(z); a=torch.sigmoid(s.ag(zz))*s.a(zz); b=torch.sigmoid(s.bg(zz))*s.b(zz)
        return z + torch.sigmoid(s.og(zz))*s.o(s.lno(triangle_update(a, b)))
class PairTransition(nn.Module):
    def __init__(s, c): super().__init__(); s.ln=nn.LayerNorm(c); s.f=nn.Sequential(nn.Linear(c,2*c),nn.GELU(),nn.Linear(2*c,c))
    def forward(s, z): return z + s.f(s.ln(z))
print('Evoformer layers ready (all from rungs 02-03).')

### Rep 1 — `symmetrize(z)`
Return the symmetric part of the pair representation: `(z + zᵀ) / 2`, where the
transpose swaps the `i` and `j` axes only (channels untouched). `z` is `[L, L, c]`.

In [ ]:
def symmetrize(z):
    '''Average z with its (i,j)->(j,i) transpose, keeping the channel axis fixed.'''
    # YOUR CODE HERE
    # hint: z.transpose(0, 1) swaps the first two axes
    raise NotImplementedError

# --- checkpoint ---
z = torch.randn(L, L, 5)
zs = symmetrize(z)
assert zs.shape == z.shape
assert torch.allclose(zs, zs.transpose(0, 1), atol=1e-6), 'output must be symmetric in i,j'
assert torch.allclose(zs[3, 7], (z[3, 7] + z[7, 3]) / 2, atol=1e-6)
print('symmetrize ok — z[i,j] and z[j,i] now agree, as a distance map must ✓')

## Part 2 — the Evoformer block

Wire the four steps. Given the layers, one block is:

```
m = col_attn( row_attn(m, z) )                     # MSA reasons (steps 1)
z = z + outer_product_mean(m)                      # MSA writes to pair (step 2)
z = pair_transition( tri_mul(z) )                  # pair reasons + transition (steps 3-4)
z = symmetrize(z)
```

That MSA→pair arrow (`outer_product_mean`) is the crux: without it, the two
representations never talk and the pair rep can't benefit from the MSA's reasoning.

### Rep 2 — `evoformer_block(m, z, row, col, opm, tri, trans)`
Apply the five given layers in the order above and return the updated `(m, z)`. Finish
with `symmetrize`.

In [ ]:
def evoformer_block(m, z, row, col, opm, tri, trans):
    '''One Evoformer block: MSA attention -> MSA->pair -> pair triangle+transition -> symmetrize.'''
    # YOUR CODE HERE
    # hint: m = col(row(m, z)); z = z + opm(m); z = trans(tri(z)); z = symmetrize(z)
    raise NotImplementedError

# --- checkpoint ---
cm, cz = 32, 32
row, col = RowAttnPairBias(cm, cz), ColAttn(cm)
opm, tri, trans = OuterProductMean(cm, cz), TriMul(cz), PairTransition(cz)
m0 = torch.randn(Nseq, L, cm); z0 = torch.randn(L, L, cz)
m1, z1 = evoformer_block(m0, z0, row, col, opm, tri, trans)
assert m1.shape == m0.shape and z1.shape == z0.shape, 'shapes must be preserved'
assert not torch.allclose(m1, m0) and not torch.allclose(z1, z0), 'both reps must update'
assert torch.allclose(z1, z1.transpose(0, 1), atol=1e-5), 'pair rep must come out symmetric'
# the MSA->pair path must be live: change the MSA, and z must respond
m2 = m0 + torch.randn_like(m0)
_, z2 = evoformer_block(m2, z0, row, col, opm, tri, trans)
assert not torch.allclose(z1, z2), 'z must depend on the MSA (the outer-product-mean arrow)'
print('evoformer block ok — m and z co-refine, pair rep stays symmetric ✓')

## Part 3 — the distogram head

Stack a few blocks, then decode `z[i,j]` into logits over the `16` distance bins. Two
readouts you will want from that distribution:

- the **expected distance** — `sum_bin  P(bin) · center(bin)` — a single calibrated
  distance per pair, useful for comparison and (later) for initializing structure;
- a **contact score** — the total probability mass in the "contact" bins.

### Rep 3 — `expected_distance(logits, centers)`
Softmax the distogram logits over the bin axis and take the expectation against
`centers`. `logits` is `[L, L, 16]`, `centers` is `[16]`. Return `[L, L]`.

In [ ]:
def expected_distance(logits, centers):
    '''Expected Ca-Ca distance per pair from distogram logits.'''
    # YOUR CODE HERE
    # hint: p = softmax(logits, dim=-1); return (p * centers).sum(-1)
    raise NotImplementedError

# --- checkpoint ---
lg = torch.zeros(L, L, 16); lg[..., 3] = 20.0     # all mass on bin 3
ed = expected_distance(lg, BIN_CENTERS)
assert ed.shape == (L, L)
assert torch.allclose(ed, BIN_CENTERS[3].expand(L, L), atol=1e-3), 'peaked distogram -> that bin center'
print('expected distance ok — a distogram becomes a calibrated distance in angstrom-like units ✓')

### Rep 4 — `contact_score(logits, is_contact_bin)`
Total probability that a pair is a contact: softmax over bins, then sum the mass in the
bins flagged by the boolean `is_contact_bin` `[16]`. Return `[L, L]`.

In [ ]:
def contact_score(logits, is_contact_bin):
    '''P(contact) per pair = summed distogram mass over the contact bins.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
p_contact = contact_score(lg, IS_CONTACT_BIN)     # lg was peaked at bin 3
assert p_contact.shape == (L, L)
expected = float(IS_CONTACT_BIN[3])               # is bin 3 a contact bin?
assert abs(p_contact[0, 0].item() - expected) < 1e-3, 'mass should land on/off the contact bins'
print('contact score ok — the distogram also yields a contact probability ✓')

## Part 4 — train the trunk, and where triangles earn their keep

We now train the full Evoformer trunk to predict the distogram, and — as the honest
control — a **per-pair** model with the same budget that maps the coevolution seed to a
distance distribution *for each pair independently*, with no cross-pair communication.

The prediction: the per-pair model will be badly beaten on **distance** prediction,
because a coherent distance map has to satisfy the triangle inequality — a fact about
triples that only the trunk's triangle operations can represent. (On plain binary
contact precision the toy trunk merely *matches* the raw APC baseline; the Evoformer's
contact-precision edge is a large-scale phenomenon. But the distances — the thing the
structure module actually needs — are where the trunk is unarguably better.)

In [ ]:
class Trunk(nn.Module):
    def __init__(self, cm=32, cz=32, nb=3, triangle=True):
        super().__init__(); self.triangle = triangle
        self.emb = nn.Embedding(Q, cm); self.pos = nn.Embedding(L, cm); self.zin = nn.Linear(1, cz)
        self.row = nn.ModuleList([RowAttnPairBias(cm, cz) for _ in range(nb)])
        self.col = nn.ModuleList([ColAttn(cm) for _ in range(nb)])
        self.opm = nn.ModuleList([OuterProductMean(cm, cz) for _ in range(nb)])
        self.tri = nn.ModuleList([TriMul(cz) for _ in range(nb)])
        self.tr = nn.ModuleList([PairTransition(cz) for _ in range(nb)])
        # per-pair baseline swaps the whole trunk for independent MLPs on z
        self.pp = nn.ModuleList([nn.Sequential(nn.LayerNorm(cz), nn.Linear(cz, 2*cz), nn.GELU(), nn.Linear(2*cz, cz)) for _ in range(nb)])
        self.head = nn.Linear(cz, 16)
    def forward(self, x, zseed):
        z = self.zin(zseed[..., None])
        if not self.triangle:                              # per-pair baseline: no MSA, no triangles
            for f in self.pp: z = z + f(z)
            return self.head(symmetrize(z))
        m = self.emb(x) + self.pos(torch.arange(L))
        for r, c, o, t, tr in zip(self.row, self.col, self.opm, self.tri, self.tr):
            m, z = evoformer_block(m, z, r, c, o, t, tr)
        return self.head(symmetrize(z))

def train_model(triangle, steps=600, seed=1):
    torch.manual_seed(seed); net = Trunk(triangle=triangle); opt = torch.optim.AdamW(net.parameters(), lr=2e-3)
    for st in range(steps):
        i = int(rng.integers(len(Mtr)))
        loss = F.cross_entropy(net(Mtr[i], Ztr[i]).reshape(-1, 16), Dtr[i].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return net

t0 = time.time()
trunk = train_model(triangle=True)
baseline = train_model(triangle=False, steps=500)
print('trained trunk + per-pair baseline in %.0fs' % (time.time() - t0))

### Rep 5 — `distogram_metrics(net)`
For the held-out set, return `(mae, top1_acc)`: the mean absolute error of the expected
distance vs the true (binned) distance, and the distogram top-1 bin accuracy. Score
only the `UM` pairs. Use `expected_distance`, `BIN_CENTERS`, and the given tensors
`Mte, Zte, Dte`.

In [ ]:
@torch.no_grad()
def distogram_metrics(net):
    '''-> (expected-distance MAE, distogram top-1 accuracy) over the held-out proteins.'''
    # YOUR CODE HERE
    # hint: for each test i: lg = net(Mte[i], Zte[i]); ed = expected_distance(lg, BIN_CENTERS)
    #   true_ed = BIN_CENTERS[Dte[i]]; mae += (ed[UM]-true_ed[UM]).abs().mean()
    #   acc += (lg.argmax(-1)[UM] == Dte[i][UM]).float().mean()
    raise NotImplementedError

# --- checkpoint ---
mae_t, acc_t = distogram_metrics(trunk)
mae_b, acc_b = distogram_metrics(baseline)
assert mae_t < mae_b - 0.3, 'the trunk should predict distances much better than per-pair'
assert acc_t > acc_b + 0.1, 'and get far more distogram bins exactly right'
print('distance MAE (bins, lower better):  trunk %.3f   per-pair %.3f' % (mae_t, mae_b))
print('distogram top-1 accuracy:           trunk %.3f   per-pair %.3f' % (acc_t, acc_b))
print('\nThe trunk predicts a globally CONSISTENT distance map; the per-pair model,')
print('lacking any triangle reasoning, cannot — exactly the rung-03 lesson, in 3D. ✓')

In [ ]:
# See it: predicted vs true distance map for a held-out protein.
i = 0
with torch.no_grad():
    ed_pred = expected_distance(trunk(Mte[i], Zte[i]), BIN_CENTERS).numpy()
    ed_pp = expected_distance(baseline(Mte[i], Zte[i]), BIN_CENTERS).numpy()
ed_true = BIN_CENTERS[Dte[i]].numpy()
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.7))
for ax, mp, ti in zip(axes, [ed_true, ed_pred, ed_pp],
                      ['true distance map', 'trunk (Evoformer)', 'per-pair baseline']):
    im = ax.imshow(mp, cmap='viridis_r', vmin=1.5, vmax=9)
    ax.set_title(ti, fontsize=10); ax.set_xlabel('residue j'); ax.set_ylabel('residue i')
fig.colorbar(im, ax=axes, fraction=0.025, label='Ca-Ca distance')
plt.show()
print('The trunk reproduces the diagonal band and the off-diagonal contacts of the true')
print('map; the per-pair baseline smears into a structureless wash. ✓')

## Reflection — what just transferred

- **The Evoformer block is four steps in a fixed order:** MSA attention → MSA-writes-to-
  pair (outer-product mean) → pair triangle reasoning → transitions. The two
  representations refine each other, block after block.
- **The MSA→pair arrow is not optional.** Zero it and `z` never learns from the MSA's
  reasoning. You verified the dependency directly.
- **The pair rep must stay symmetric** — it encodes a symmetric physical quantity, the
  distance — so we symmetrize after every pair update.
- **The distogram is the trunk's geometric output:** a distribution over binned
  distances per pair, from which you read a calibrated expected distance and a contact
  probability. This is what the structure module consumes.
- **Distances are where triangles pay off.** A per-pair model can score contacts
  passably but cannot predict a *coherent distance map*, because distances obey the
  triangle inequality — a constraint over triples. Your trunk beat it ~2× on distance
  error. (Honestly: on binary contacts the toy trunk only ties raw APC — the
  Evoformer's contact-precision advantage is a scale phenomenon, not a toy one.)

**Track A is complete.** You have a trunk that turns an MSA into a trustworthy,
geometrically-consistent pair representation and a distance map. **Next rung:**
`AF2·05 — Residue frames & the structure module` — we stop describing geometry with
pairwise distances and start *placing atoms in space*, using the SE(3) frames you first
met in `0.1` and `1.6`.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def symmetrize(z):
    return 0.5 * (z + z.transpose(0, 1))

def evoformer_block(m, z, row, col, opm, tri, trans):
    m = col(row(m, z))                # MSA reasons about itself (row uses the pair bias)
    z = z + opm(m)                    # MSA writes into the pair representation
    z = trans(tri(z))                 # pair reasons (triangles) + transition
    return m, symmetrize(z)

def expected_distance(logits, centers):
    return (F.softmax(logits, dim=-1) * centers).sum(-1)

def contact_score(logits, is_contact_bin):
    return F.softmax(logits, dim=-1)[..., is_contact_bin].sum(-1)

@torch.no_grad()
def distogram_metrics(net):
    mae, acc = 0.0, 0.0
    for i in range(len(Mte)):
        lg = net(Mte[i], Zte[i]); ed = expected_distance(lg, BIN_CENTERS)
        true_ed = BIN_CENTERS[Dte[i]]
        mae += (ed[UM] - true_ed[UM]).abs().mean().item()
        acc += (lg.argmax(-1)[UM] == Dte[i][UM]).float().mean().item()
    return mae / len(Mte), acc / len(Mte)

print('reference solutions loaded — re-run the checkpoint cells above')